# Equity Classifier Scale Benchmark

Measure whether the classifier-only equity signal pipeline is fast enough to support daily retraining for an options-first workflow.

This notebook uses a daily-style benchmark: train on all available pre-2020 labels, but score only a recent window. That reflects the live job we actually care about; a daily retrain should not rescore every historical out-of-sample date from 2020 onward.

Native Zipline execution, MLflow logging, and sampled `backtesting.py` validation are disabled here so the timing isolates feature loading, label loading, CUDA RF training, and recent signal scoring. Full execution-engine validation belongs in selected research runs after a universe and model setting look worth keeping.

The default benchmark runs 1T and 100B universes end-to-end. A 10B run is intentionally not part of the default because the current pipeline still spends too long in feature/label preparation; 10B should become a Dagster/warehouse materialized-label workflow before it is treated as a daily retrain target.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
from time import perf_counter

import pandas as pd
from IPython.display import display, Markdown

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
for candidate in reversed((REPO_ROOT, REPO_ROOT.parent / 'quant-warehouse')):
    if candidate.exists():
        candidate_str = str(candidate)
        sys.path[:] = [entry for entry in sys.path if entry != candidate_str]
        sys.path.insert(0, candidate_str)

from quant_orchestrator.research_tools import MLTradingExperimentConfig, run_ml_trading_experiment

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)

In [ ]:
CAPS = [
    ('1T', 1_000_000_000_000),
    ('100B', 100_000_000_000),
    # 10B is the likely options universe, but it is not daily-friendly yet without
    # materialized feature panels and oracle labels. Run manually after those assets exist.
    # ('10B', 10_000_000_000),
]

BASE_CONFIG = dict(
    mode='classifier',
    start_date='1900-01-01',
    end_date=None,
    train_end='2019-12-31',
    oos_start='2020-01-01',
    score_start='2026-01-01',
    top_k_values=(5,),
    run_zipline_backtests=False,
    include_yearly_vectorized_diagnostics=True,
    backtesting_py_symbol_cases_per_side=0,
    log_mlflow=False,
)

# Daily retrain profile: keep the GPU RF, but avoid the heavier research profile.
RF_PARAMS = {
    'n_estimators': 100,
    'max_depth': 12,
    'max_features': 'sqrt',
    'n_bins': 64,
    'n_streams': 8,
}


## Run Scale Benchmark

In [ ]:
results = []
phase_frames = []
robustness_frames = []
validation_frames = []

for label, cap in CAPS:
    print(f'Running {label} cap={cap:,}')
    started = perf_counter()
    config = MLTradingExperimentConfig(
        experiment_name=f'equity_classifier_scale_{label.lower()}_diagnostics',
        min_market_cap=cap,
        rf_params=RF_PARAMS,
        **BASE_CONFIG,
    )
    result = run_ml_trading_experiment(config)
    wall_seconds = perf_counter() - started
    row = {
        'cap_label': label,
        'min_market_cap': cap,
        'wall_seconds': wall_seconds,
        **result.metrics,
        'model_rows': len(result.model_results),
        'score_rows': len(result.strategy_scores),
        'strategy_sources': result.strategy_scores['strategy_source'].nunique() if not result.strategy_scores.empty else 0,
        'symbol_robustness_rows': len(result.symbol_robustness_summary),
        'backtesting_py_validation_rows': len(result.backtesting_py_symbol_validation),
        'mlflow_run_id': result.mlflow_run_id,
    }
    results.append(row)
    phase = result.phase_timings.copy()
    phase.insert(0, 'cap_label', label)
    phase_frames.append(phase)
    robust = result.symbol_robustness_summary.copy()
    robust.insert(0, 'cap_label', label)
    robustness_frames.append(robust)
    validation = result.backtesting_py_symbol_validation.copy()
    validation.insert(0, 'cap_label', label)
    validation_frames.append(validation)
    display(pd.DataFrame([row]))

benchmark_summary = pd.DataFrame(results)
phase_timings = pd.concat(phase_frames, ignore_index=True) if phase_frames else pd.DataFrame()
robustness = pd.concat(robustness_frames, ignore_index=True) if robustness_frames else pd.DataFrame()
validation = pd.concat(validation_frames, ignore_index=True) if validation_frames else pd.DataFrame()

display(benchmark_summary)

## Runtime Breakdown

In [ ]:
phase_pivot = (
    phase_timings
    .pivot_table(index='phase', columns='cap_label', values='seconds', aggfunc='sum')
    .reindex(columns=[label for label, _ in CAPS])
)
display(phase_pivot)

display(
    phase_timings
    .sort_values(['cap_label', 'seconds'], ascending=[True, False])
    .groupby('cap_label', as_index=False)
    .head(8)
)

## Robustness Leaders

In [ ]:
leaders = (
    robustness
    .sort_values(['cap_label', 'beat_buy_hold_rate', 'median_excess_total_return', 'median_strategy_sharpe'], ascending=[True, False, False, False])
    .groupby('cap_label', as_index=False)
    .head(10)
)
display(leaders)

display(validation.sort_values(['cap_label', 'excess_total_return'], ascending=[True, False]))

## Written Analysis

In [ ]:
lines = ['### Scale Benchmark Analysis', '']
if not benchmark_summary.empty:
    for row in benchmark_summary.itertuples(index=False):
        lines.append(
            f'- {row.cap_label}: {row.symbols} symbols, {row.trained_models} trained models, '
            f'{row.score_rows:,} score rows, wall time {row.wall_seconds:.1f}s.'
        )
    fastest = benchmark_summary.sort_values('wall_seconds').iloc[0]
    largest = benchmark_summary.sort_values('min_market_cap').iloc[0]
    lines.append('')
    lines.append('- Use diagnostics-only runs for daily retrain checks; reserve native Zipline for selected candidates or scheduled validation.')
    lines.append('- For options trading, 10B is the practical upper universe only if the daily retrain wall time stays psychologically acceptable and ThetaData coverage is liquid enough.')
    if 'train_family_models' in set(phase_timings['phase']):
        train_phase = phase_timings.loc[phase_timings['phase'].eq('train_family_models')]
        slow_train = train_phase.sort_values('seconds', ascending=False).head(1)
        if not slow_train.empty:
            r = slow_train.iloc[0]
            lines.append(f'- Slowest RF phase observed: {r["cap_label"]} train_family_models at {r["seconds"]:.1f}s.')
    if not robustness.empty:
        best = robustness.sort_values(['beat_buy_hold_rate', 'median_excess_total_return'], ascending=False).iloc[0]
        lines.append(
            f'- Best robustness row: {best["cap_label"]} {best["strategy_source"]} / {best["variant"]} '
            f'beat buy-and-hold on {best["beat_buy_hold_rate"]:.1%} of symbols.'
        )

display(Markdown('
'.join(lines)))